# Policy evaluation on the microfinance villages

For each of the 43 Karnataka villages: detects communities, simulates diffusion, fits the group Ising model (logistic, EMVS), trains a Q-Ising (CQL) agent and compares it with random, degree, LIR and degree-bin policies; also EMVS fit diagnostics and seeding illustrations.

**Reads** `data/villages/adjacency_all.json`. **Writes** `village_policy_results.csv` and figures in `figures/` (paper figures come from `plotting/`).

Run with this folder as working directory, after `pip install -e .` from the repository root.

# Edge-betweenness community Q-Ising on villages

Bins are igraph edge-betweenness communities sorted by size (bin 0 = largest); rates by rank: bin 0 low-spreader/high-churner (β=0.01), bin 1 high-spreader (β=0.25), the rest followers. Per village: logistic fit, EMVS, CQL (no skipping); MCMC and ensemble on a development village, then a loop over all.

In [ ]:
import os, sys, shutil, tempfile, glob as _glob, json, time as _time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

MF_DIR  = os.path.dirname(os.path.abspath("policy_eval.ipynb"))

import igraph as ig
import d3rlpy
from qising.baselines import RandomPolicy, DegreeCentralityPolicy, LIRPolicy, DegreeBinPolicy, QIsingPolicy
from qising.environments.edge_bet_env import EdgeBetCommunitySISEnv
from qising.ising.group_ising import fit_group_ising_emvs, predict_group_ising_emvs
from qising.utils import build_village_envdata, run_policy_episodes
import logging; logging.getLogger("d3rlpy").setLevel(logging.WARNING)

ADJ_JSON = os.path.join(MF_DIR, "..", "..", "data", "villages", "adjacency_all.json")
FIG_DIR  = os.path.join(MF_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=150, bbox_inches="tight")

# ── Beta/delta lists assigned by community size rank (0 = largest) ────────
BETA_LIST  = [0.01, 0.5, 0.05, 0.07,  0.06, 0.02,  0.01,  0.4,  0.1, 0.3]
DELTA_LIST = [0.5, 0.9, 0.9, 0.6, 0.5,  0.50,  0.7,  0.6,  0.5, 0.8]

# ── Community detection ──────────────────────────────
COMM_METHOD   = "edge_betweenness"  # "louvain" | "edge_betweenness"
MIN_COMM_SIZE = 10

# ── Episode / training hyperparameters ─────────────────
T_TRAIN        = 500
N_TRAIN_EPS    = 1
DATA_SEED      = 42
INIT_RATE      = 0.0
TEST_INIT_RATE = 0.0
T_TEST         = 25
K_SEEDS        = T_TEST
N_TEST_EPS     = 50
SKIP_PROB      = 0

# ── CQL ─────────────────────────────────────────────────────
DISCOUNT          = 0.8
ALPHA_CQL         = 0.1


# ── Evaluation mode ────────────────────────────────
# True  → each test episode starts from the adoption state at the END of
#          the first behavioural trajectory (policy takes over from there).
# False → each test episode starts from scratch (init_rate / clean slate).
USE_BEHAVIORAL_SNAPSHOT = False

POLICY_DISPLAY = {
    "degree": "Degree", "lir": "LIR",
    "degree_bin": "Degree-Bin", "plain_dqn": "Plain DQN", "q_ising": "Q-Ising",
    "q_ising_ensemble": "Ensemble Q-Ising",
}
POLICY_COLORS = {
    "degree": "#e67e22", "lir": "#9b59b6",
    "degree_bin": "#f39c12", "plain_dqn": "#3498db", "q_ising": "#27ae60",
    "q_ising_ensemble": "#e74c3c",
}

print("Imports OK.", f"d3rlpy {d3rlpy.__version__}")

In [ ]:
# ── Load all adjacency matrices and detect communities ───────────────────
adj_all  = json.load(open(ADJ_JSON))
VILL_IDS = sorted(adj_all.keys(), key=lambda x: int(x))

print(f"Detecting communities for {len(VILL_IDS)} villages  [method={COMM_METHOD}] ...")
_t0 = _time.perf_counter()
village_data = {}
failed_vills = []
for v in VILL_IDS:
    A = np.array(adj_all[v], dtype=float); np.fill_diagonal(A, 0)
    try:
        village_data[v] = build_village_envdata(
            A, BETA_LIST, DELTA_LIST,
            comm_method=COMM_METHOD, min_comm_size=MIN_COMM_SIZE)
    except Exception as e:
        failed_vills.append((v, str(e)))

if failed_vills:
    print(f"  Failed: {failed_vills}")

# ── Require at least 2 communities (single-community villages are trivial) ─
single_comm = [v for v in village_data if village_data[v]["n_bins"] < 2]
if single_comm:
    print(f"  Dropping {len(single_comm)} village(s) with < 2 communities: {single_comm}")
    for v in single_comm:
        del village_data[v]

VIABLE_VILLS = list(village_data.keys())
print(f"Done ({_time.perf_counter()-_t0:.1f}s)  "
      f"Viable: {len(VIABLE_VILLS)} / {len(VILL_IDS)}  "
      f"(dropped {len(single_comm)} single-community)")

In [ ]:
# R-value check per bin across all villages
from collections import defaultdict
r_by_rank = defaultdict(list)
for v in VIABLE_VILLS:
    vd = village_data[v]
    for b in range(vd["n_bins"]):
        m = np.where(vd["block_labels"] == b)[0]
        if len(m) == 0: continue
        k_bar = vd["adj"][m].sum(axis=1).mean()
        r = vd["beta_vec"][m].mean() * k_bar / vd["delta_vec"][m].mean()
        r_by_rank[b].append(r)

print("=== R = beta*k_bar/delta by bin rank (across all villages) ===")
for b in sorted(r_by_rank.keys()):
    rs = np.array(r_by_rank[b])
    label = "SUPERCRIT" if rs.mean() > 1 else "subcrit"
    print(f"  Bin {b}  beta={BETA_LIST[min(b,len(BETA_LIST)-1)]:.3f}  "
          f"R mean={rs.mean():.2f}  min={rs.min():.2f}  max={rs.max():.2f}  [{label}]")

In [ ]:
# ── Histogram: number of communities per village ─────────────────────────
cluster_counts = [village_data[v]["n_bins"] for v in VIABLE_VILLS]
all_bin_sizes  = [sz for v in VIABLE_VILLS for sz in village_data[v]["bin_counts"].tolist()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
bins_h = np.arange(0.5, max(cluster_counts) + 1.5, 1)
ax.hist(cluster_counts, bins=bins_h, edgecolor="white", color="steelblue", alpha=0.85)
ax.set_xlabel("Number of communities detected")
ax.set_ylabel("Number of villages")
ax.set_title("Community count distribution (edge-betweenness)")
ax.set_xticks(np.arange(1, max(cluster_counts) + 1))

ax = axes[1]
ax.hist(all_bin_sizes, bins=20, edgecolor="white", color="coral", alpha=0.85)
ax.set_xlabel("Nodes per community")
ax.set_ylabel("Count")
ax.set_title("Nodes per community (all villages)")

plt.tight_layout()
savefig(fig, "community_histogram")
plt.show()

print("=== Community count per village ===")
print(f"  Mean: {np.mean(cluster_counts):.1f}  Median: {np.median(cluster_counts):.1f}  ")
print(f"  Min:  {np.min(cluster_counts)}        Max: {np.max(cluster_counts)}")
print("=== Nodes per community (pooled) ===")
print(f"  Mean: {np.mean(all_bin_sizes):.1f}  Median: {np.median(all_bin_sizes):.1f}")
print(f"  Min:  {np.min(all_bin_sizes)}        Max: {np.max(all_bin_sizes)}")

## All villages: training and testing

One CQL agent per village (Q-Ising EMVS and plain DQN).

In [ ]:
# ── Per-village train + test loop ────────────────────────────────────
all_village_results  = {}
all_village_curves   = {}
LOOP_EVAL_SEED = 77
_t_loop_start  = _time.perf_counter()

_qi_kwargs = dict(discount=DISCOUNT, alpha_cql=ALPHA_CQL)

for _v in VIABLE_VILLS:
    print(f"\n{'='*50}  Village {_v}")
    _vd   = village_data[_v]
    _nb_v = _vd["n_bins"]

    # ── 1. Collect behavioral trajectories ───────────────────────
    _trajs_v = []
    for _ep in range(N_TRAIN_EPS):
        _env_ep = EdgeBetCommunitySISEnv(
            _vd["adj"], _vd["beta_vec"], _vd["delta_vec"], _vd["block_labels"],
            T=T_TRAIN, seed=DATA_SEED + _ep, init_rate=INIT_RATE)
        _trajs_v.append(_env_ep.collect_behavioral_episode(skip_prob=SKIP_PROB))

    _snapshot_v = _trajs_v[0]["adopted"][-1].copy() if USE_BEHAVIORAL_SNAPSHOT else None

    # ── 2. Train Q-Ising EMVS and Plain DQN policies ─────────────────
    _qi_pol_v = QIsingPolicy(
        ising_augmentation=True, label=f"v{_v}_emvs", **_qi_kwargs
    ).fit(_trajs_v, _vd["adj"], _vd["block_labels"])

    _dqn_pol_v = QIsingPolicy(
        ising_augmentation=False, label=f"v{_v}_bin", **_qi_kwargs
    ).fit(_trajs_v, _vd["adj"], _vd["block_labels"])

    print(f"  n_bins={_nb_v}  trained CQL agents")

    # ── 3. Evaluate ───────────────────────────────────────────────
    _norm_v = T_TEST * _vd["N"]

    def _make_te_v(seed, _vd=_vd):
        return EdgeBetCommunitySISEnv(
            _vd["adj"], _vd["beta_vec"], _vd["delta_vec"], _vd["block_labels"],
            T=T_TEST, seed=seed, init_rate=TEST_INIT_RATE)

    _ref_v_e   = _make_te_v(seed=0)
    _deg_v     = DegreeCentralityPolicy(_ref_v_e)
    _lir_v     = LIRPolicy(_ref_v_e)
    _deg_bin_v = DegreeBinPolicy(_ref_v_e)

    _pols_v = [
        ("random",     RandomPolicy(_nb_v),     None),
        ("degree",     _deg_v,                  _deg_v.reset),
        ("lir",        _lir_v,                  _lir_v.reset),
        ("degree_bin", _deg_bin_v,              _deg_bin_v.reset),
        ("plain_dqn",  _dqn_pol_v,              None),
        ("q_ising",    _qi_pol_v,               None),
    ]

    all_village_results[_v] = {"N": _vd["N"], "n_bins": _nb_v}
    all_village_curves[_v]  = {}
    for pname, pfn, rst in _pols_v:
        tots_v, crvs_v, _ = run_policy_episodes(
            pfn, _make_te_v, N_TEST_EPS, K_SEEDS, _norm_v, LOOP_EVAL_SEED, rst, init_state=_snapshot_v)
        all_village_results[_v][pname] = (tots_v.mean(), tots_v.std(),
                                          tots_v.std() / np.sqrt(N_TEST_EPS))
        all_village_curves[_v][pname]  = crvs_v
        print(f"    {pname:<12} mean={tots_v.mean():.4f}  SE={tots_v.std()/np.sqrt(N_TEST_EPS):.4f}")

print(f"\nAll villages done. Total time: {(_time.perf_counter()-_t_loop_start)/60:.1f} min")

In [ ]:
# ── Aggregate results across all villages ────────────────────────────────
pol_names = ["random", "degree", "lir", "degree_bin", "plain_dqn", "q_ising"]

rows = []
for _v in VIABLE_VILLS:
    if _v not in all_village_results: continue
    r = all_village_results[_v]
    row = {"village": _v, "N": r["N"], "n_bins": r["n_bins"]}
    for p in pol_names:
        if p in r:
            row[p + "_mean"] = r[p][0]
            row[p + "_se"]   = r[p][2]
    rows.append(row)

df = pd.DataFrame(rows).set_index("village")

# Win rate over LIR
for p in ["degree", "lir", "degree_bin", "plain_dqn", "q_ising"]:
    col = p + "_mean"
    if col in df.columns and "lir_mean" in df.columns:
        wins = (df[col] > df["lir_mean"]).sum()
        print(f"  {POLICY_DISPLAY.get(p, p):<22} wins over LIR: {wins}/{len(df)} "              f"({100*wins/len(df):.0f}%)")

print("=== Mean adoption (macro-averaged over villages) ===")
for p in pol_names:
    col = p + "_mean"
    if col in df.columns:
        print(f"  {POLICY_DISPLAY.get(p, p):<22} {df[col].mean():.4f} ± {df[col].std():.4f}")

In [ ]:
#Improvement graph
#get the best alternative
df["best_alternative"] = df[["degree_bin_mean"]].max(axis=1)
df["Improvement"] = (df["q_ising_mean"] - df["best_alternative"]) / (df["best_alternative"].abs() + 1e-9) * 100
df = df.sort_values('Improvement').reset_index(drop=True)

# 2. Define Visual Style 
def get_color(val):
    if val < 0:
        return "#C2510B" # Steel Blue for negative
    elif 0 <= val <= 5:
        return '#808080' # Grey for 0-5% (below threshold)
    else:
        return "#0577BE" # Dark Red for > 5%

colors = [get_color(x) for x in df['Improvement']]

# 3. Create the Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Use edgecolor='black' to get the sharp outlines from your image
ax.bar(df.index, df['Improvement'], color=colors, edgecolor='black', linewidth=0.7)

# Threshold Lines (+/- 5%)
ax.axhline(5, color='darkred', linestyle='--', linewidth=1.2, label='+5% threshold')
ax.axhline(-5, color='darkred', linestyle='--', linewidth=1.2, label='-5% threshold')
ax.axhline(0, color='black', linewidth=1) 

# Formatting & Grid
ax.yaxis.grid(True, which='both', linestyle='-', alpha=0.2) # Show grid for log levels
ax.set_axisbelow(True)

ticks = [-20, -5, 0, 5, 20, 50]
ax.set_yticks(ticks)
ax.set_yticklabels([f'{t}%' for t in ticks])

ax.set_title('Q-Ising Improvement over Degree Bin', fontsize=15)
ax.set_xlabel('Villages (sorted by improvement)', fontsize=12)
ax.set_ylabel('% improvement over degree bin', fontsize=12)
ax.legend(loc='upper left')

plt.tight_layout()
savefig(fig, "q_ising_improvement_hist")
plt.show()

In [ ]:
#Compare 1 std intervals of Q-Ising vs others
#Better means that the lower bound of the other policy's 1 SE interval is above the upper bound of Q-Ising's 1 SE interval, or vice versa.
print("\n=== Q-Ising vs others (1 SE interval) ===")
for p in pol_names:
    if p == "q_ising": continue
    col_mean = p + "_mean"
    col_se   = p + "_se"
    if col_mean in df.columns and "q_ising_mean" in df.columns and "q_ising_se" in df.columns:
        q_mean = df["q_ising_mean"]
        q_se   = df["q_ising_se"]
        p_mean = df[col_mean]
        p_se   = df[col_se]
        q_lower = q_mean - q_se
        q_upper = q_mean + q_se
        p_lower = p_mean - p_se
        p_upper = p_mean + p_se
        better_other = ((p_lower > q_upper)).sum()
        better_qising = ((q_lower > p_upper)).sum()
        print(f"  {POLICY_DISPLAY.get(p, p):<22} better than Q-Ising: {better_other}/{len(df)} "              f"({100*better_other/len(df):.0f}%)")
        print(f"  {POLICY_DISPLAY.get(p, p):<22} worse than Q-Ising: {better_qising}/{len(df)} "              f"({100*better_qising/len(df):.0f}%)")

In [ ]:
#Save df to CSV for later analysis
df.to_csv(os.path.join("village_policy_results.csv"), index=True)

In [ ]:
# ── LaTeX longtable: per-village policy results ───────────────────────────
pol_order   = ["random", "degree", "lir", "degree_bin", "plain_dqn", "q_ising"]
col_headers = [POLICY_DISPLAY.get(p, p) for p in pol_order]
ncols       = len(pol_order)

def _fmt_cell(mean, se, bold):
    s = f"{mean:.3f} ({se:.3f})"
    return r"\textbf{" + s + "}" if bold else s

lines = []
lines.append(r"\begin{longtable}{l" + "c" * ncols + "}")
lines.append(
    r"\caption{Per-village policy evaluation. Mean (SE). "
    r"Bold: policy upper bound (mean + SE) exceeds best policy lower bound (mean $-$ SE).} \\"
)
lines.append(r"\label{tab:village_results} \\")
lines.append(r"\toprule")
lines.append("Village & " + " & ".join(col_headers) + r" \\")
lines.append(r"\midrule")
lines.append(r"\endfirsthead")
lines.append(
    r"\multicolumn{" + str(ncols + 1) + r"}{c}{\tablename\ \thetable{} (continued)} \\"
)
lines.append(r"\toprule")
lines.append("Village & " + " & ".join(col_headers) + r" \\")
lines.append(r"\midrule")
lines.append(r"\endhead")
lines.append(r"\midrule")
lines.append(
    r"\multicolumn{" + str(ncols + 1) + r"}{r}{Continued on next page} \\"
)
lines.append(r"\endfoot")
lines.append(r"\bottomrule")
lines.append(r"\endlastfoot")

for _v, row in df.iterrows():
    # best policy per village = highest mean; use its lower bound as threshold
    best_mean = max(row[p + "_mean"] for p in pol_order if p + "_mean" in row)
    best_se   = next(row[p + "_se"] for p in pol_order if row[p + "_mean"] == best_mean)
    best_lb   = best_mean - best_se

    cells = [str(_v)]
    for p in pol_order:
        m_col, se_col = p + "_mean", p + "_se"
        if m_col not in row or pd.isna(row[m_col]):
            cells.append("--")
            continue
        pmean, pse = row[m_col], row[se_col]
        bold = (pmean + pse) >= best_lb
        cells.append(_fmt_cell(pmean, pse, bold))
    lines.append(" & ".join(cells) + r" \\")

lines.append(r"\end{longtable}")

latex_str = "\n".join(lines)
print(latex_str)

## What Drives Q-Ising's Improvement over LIR?

In [ ]:
# ── What drives Q-Ising improvement over LIR? ─ feature extraction ───────
from scipy.stats import spearmanr
from scipy.stats import entropy as _entropy

feat_rows = []
for _v in VIABLE_VILLS:
    if _v not in all_village_results or _v not in village_data:
        continue
    r     = all_village_results[_v]
    vd    = village_data[_v]
    bc    = vd["bin_counts"].astype(float)
    N_v   = float(vd["N"])
    G_v   = int(vd["n_bins"])
    adj_v = vd["adj"]
    bl    = vd["block_labels"]

    # ── existing structural features ──────────────────────────────────────
    largest_frac      = bc[0] / N_v
    size_cv           = bc.std() / (bc.mean() + 1e-9)
    probs             = bc / bc.sum()
    size_entropy      = float(_entropy(probs))
    max_entropy       = np.log(G_v) if G_v > 1 else 1.0
    size_entropy_norm = size_entropy / max_entropy

    deg        = adj_v.sum(axis=1)
    avg_degree = float(deg.mean())
    deg_cv     = float(deg.std() / (deg.mean() + 1e-9))
    bin1_frac  = (bc[1] / N_v) if G_v >= 2 else 0.0

    # ── NEW: graph density (scale-free sparsity) ──────────────────────────
    density = float(adj_v.sum() / (N_v * (N_v - 1) + 1e-9))

    # ── NEW: modularity Q ─────────────────────────────────────────────────
    m_edges   = adj_v.sum() / 2
    same_comm = (bl[:, None] == bl[None, :]).astype(float)
    Q_mod     = float(
        ((adj_v - np.outer(deg, deg) / (2 * m_edges + 1e-9)) * same_comm).sum()
        / (2 * m_edges + 1e-9)
    )

    diff_comm       = (bl[:, None] != bl[None, :]).astype(float)
    inter_edge_frac = float((adj_v * diff_comm).sum() / (adj_v.sum() + 1e-9))

    # ── NEW: R_eff for bin 1 (dynamical sparsity of the high-spreader bin) ─
    bin1_mask = (bl == 1)
    if bin1_mask.sum() > 0:
        beta_b1  = float(vd["beta_vec"][bin1_mask].mean())
        delta_b1 = float(vd["delta_vec"][bin1_mask].mean())
        k_b1     = float(deg[bin1_mask].mean())
        r_eff_b1 = beta_b1 * k_b1 / (delta_b1 + 1e-9)
    else:
        r_eff_b1 = 0.0

    # ── performance ───────────────────────────────────────────────────────
    qi      = r["q_ising"][0]
    db     = r["degree_bin"][0]
    imp_abs = qi - db
    imp_pct = imp_abs / (abs(db) + 1e-9) * 100

    feat_rows.append(dict(
        village=_v, N=int(N_v), n_bins=G_v,
        largest_frac=largest_frac, bin1_frac=bin1_frac,
        size_cv=size_cv, size_entropy_norm=size_entropy_norm,
        avg_degree=avg_degree, deg_cv=deg_cv,
        density=density, Q_mod=Q_mod,
        inter_edge_frac=inter_edge_frac, r_eff_b1=r_eff_b1,
        q_ising=qi, degree_bin=db,
        imp_abs=imp_abs, imp_pct=imp_pct,
    ))

df_feat = pd.DataFrame(feat_rows).set_index("village")


In [ ]:
# ── Diagnostic plots: what predicts Q-Ising improvement? ─────────────────
feat_labels = {
    "n_bins":             "# communities",
    "largest_frac":       "Largest bin fraction",
    "bin1_frac":          "Bin-1 fraction (high-β spreader)",
    "size_cv":            "Bin-size CV",
    "size_entropy_norm":  "Bin-size entropy (norm.)",
    "avg_degree":         "Average degree",
    "deg_cv":             "Degree CV",
    "N":                  "Village size (N)",
    "density":            "Graph density",
    "Q_mod":              "Modularity Q",
    "inter_edge_frac":    "Inter-community edge frac.",
    "r_eff_b1":           "R_eff bin-1 (high-β spreader)",
}
feats = list(feat_labels.keys())

def _spearman_grid(df, y_col, y_lab, title, fname):
    ncols = 4
    nrows = int(np.ceil(len(feats) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.8 * nrows))
    axes = axes.flatten()
    rows = []
    for ax, feat in zip(axes, feats):
        x = df[feat].values
        y = df[y_col].values
        r_s, p_s = spearmanr(x, y)
        rows.append(dict(feature=feat_labels[feat], rho=round(r_s, 3), p=round(p_s, 3)))
        sc = ax.scatter(x, y, c=df["n_bins"], cmap="viridis",
                        alpha=0.75, edgecolors="white", linewidths=0.4, s=60)
        ax.axhline(0, color="#9b59b6", lw=1, ls="--", alpha=0.6)
        if x.std() > 1e-9:
            m, b = np.polyfit(x, y, 1)
            xs = np.linspace(x.min(), x.max(), 100)
            ax.plot(xs, m * xs + b, color="#e74c3c", lw=1.4, alpha=0.8)
        ax.set_xlabel(feat_labels[feat], fontsize=9)
        ax.set_ylabel(y_lab if ax is axes[0] else "", fontsize=8)
        ax.set_title(f"ρ={r_s:.2f}  p={p_s:.3f}", fontsize=9)
        ax.tick_params(labelsize=8)
    plt.colorbar(sc, ax=axes[:len(feats)], label="# communities", shrink=0.6)
    for ax in axes[len(feats):]:
        ax.set_visible(False)
    plt.suptitle(title, fontsize=11, y=1.01)
    plt.tight_layout()
    savefig(fig, fname)
    plt.show()
    return pd.DataFrame(rows)

# ── vs LIR ────────────────────────────────────────────────────────────────
df_sp_lir = _spearman_grid(
    df_feat, "imp_pct", "Q-Ising improvement over LIR (%)",
    "Q-Ising improvement over LIR vs community structure features",
    "improvement_drivers_lir_edgebet",
)

# ── Combined Spearman table ────────────────────────────────────────────────
def _sig(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

df_sp_lir["sig_LIR"] = df_sp_lir["p"].apply(_sig)

df_combined = (
    df_sp_lir.rename(columns={"rho": "ρ_LIR", "p": "p_LIR"})
    .sort_values("ρ_LIR", key=abs, ascending=False)
)
print("\nSpearman correlations: Q-Ising improvement vs LIR")
display(df_combined[["feature", "ρ_LIR", "sig_LIR", "p_LIR"]].reset_index(drop=True))

# ── High vs low modularity comparison ─────────────────────────────────────
q25_mod = df_feat["Q_mod"].quantile(0.01)
q75_mod = df_feat["Q_mod"].quantile(0.99)
hi_mod  = df_feat[df_feat["Q_mod"] >= q75_mod]
lo_mod  = df_feat[df_feat["Q_mod"] <= q25_mod]

print(f"\nHigh modularity (Q_mod ≥ {q75_mod:.3f})  n={len(hi_mod)}")
print(f"Low  modularity (Q_mod ≤ {q25_mod:.3f})  n={len(lo_mod)}")

cmp_feats = ["n_bins", "largest_frac", "bin1_frac", "size_cv",
             "size_entropy_norm", "avg_degree", "deg_cv", "N",
             "density", "Q_mod", "inter_edge_frac", "r_eff_b1",
             "imp_pct"]
cmp = pd.DataFrame({
    "high_Q_mod": hi_mod[cmp_feats].mean(),
    "low_Q_mod" : lo_mod[cmp_feats].mean(),
}).T.round(3)
cmp.index = [f"High modularity (n={len(hi_mod)})", f"Low modularity (n={len(lo_mod)})"]
print("Mean feature values and improvement by modularity group:")
display(cmp)


In [ ]:
# ── Select contrasting pair + Q_mod vs improvement scatter ───────────────
# Pair criteria: same n_bins, high Q_mod↔low improvement vs low Q_mod↔high improvement
from scipy.stats import spearmanr, pearsonr

# ── Set to an integer (e.g. 3 or 5) to pin a specific n_bins group,
FIXED_N_BINS = 8

_best_score, _best_bins = -1, None
_hi_mod_v, _lo_mod_v = None, None

_search_bins = [FIXED_N_BINS] if FIXED_N_BINS is not None else df_feat["n_bins"].unique()

for _nb in _search_bins:
    _sub = df_feat[df_feat["n_bins"] == _nb]
    if len(_sub) < 2:
        raise ValueError(f"n_bins={_nb} has only {len(_sub)} village(s) — need at least 2")
    _hi_cands = _sub[(_sub["Q_mod"] > _sub["Q_mod"].median())]
    _lo_cands = _sub[(_sub["Q_mod"] <= _sub["Q_mod"].median())]
    if _hi_cands.empty or _lo_cands.empty:
        if FIXED_N_BINS is not None:
            raise ValueError(
                f"n_bins={_nb}: cannot find one village on each side of both Q_mod "
                f"\nQ_mod range: {_sub['Q_mod'].min():.3f}–{_sub['Q_mod'].max():.3f}  "
                f"imp_pct range: {_sub['imp_pct'].min():.1f}–{_sub['imp_pct'].max():.1f}%\n"
                f"Try a different FIXED_N_BINS or set it to None for auto-selection."
            )
        continue

#Randomly select a high and low modularity village from the best n_bins group
#seed
np.random.seed(0)
_hi_mod_v   = np.random.choice(_hi_cands.index)                
_lo_mod_v   = np.random.choice(_lo_cands.index)

print(f"Contrasting pair  (n_bins={_best_bins}  score={_best_score:.3f})")
print(f"  High-Q : village {_hi_mod_v}  "
      f"Q={df_feat.loc[_hi_mod_v,'Q_mod']:.3f}  imp={df_feat.loc[_hi_mod_v,'imp_pct']:+.1f}%")
print(f"  Low-Q  : village {_lo_mod_v}  "
      f"Q={df_feat.loc[_lo_mod_v,'Q_mod']:.3f}  imp={df_feat.loc[_lo_mod_v,'imp_pct']:+.1f}%")

# ── Scatter: Q_mod vs imp_pct ─────────────────────────────────────────────
r_s, p_s = spearmanr(df_feat["Q_mod"], df_feat["imp_pct"])
r_p, _   = pearsonr(df_feat["Q_mod"],  df_feat["imp_pct"])
m_ols, b_ols = np.polyfit(df_feat["Q_mod"], df_feat["imp_pct"], 1)

fig, ax = plt.subplots(figsize=(7, 5))

sc = ax.scatter(df_feat["Q_mod"], df_feat["imp_pct"],
                c=df_feat["n_bins"], cmap="viridis",
                alpha=0.55, edgecolors="white", linewidths=0.4, s=55, zorder=2)
plt.colorbar(sc, ax=ax, label="# communities")

xs = np.linspace(df_feat["Q_mod"].min(), df_feat["Q_mod"].max(), 200)
ax.plot(xs, m_ols * xs + b_ols, color="#e74c3c", lw=1.8, alpha=0.85,
        label=f"OLS  r={r_p:.2f}")
ax.axhline(0, color="#9b59b6", lw=1, ls="--", alpha=0.5)

for _v, clr, mk, lbl in [
    (_hi_mod_v, "#e74c3c", "^", f"Vil. {_hi_mod_v} — high Q, low imp"),
    (_lo_mod_v, "#2ecc71", "v", f"Vil. {_lo_mod_v} — low Q, high imp"),
]:
    _r = df_feat.loc[_v]
    ax.scatter(_r["Q_mod"], _r["imp_pct"],
               color=clr, marker=mk, s=200, zorder=5,
               edgecolors="black", linewidths=0.9, label=lbl)
    ax.annotate(
        f"Vil. {_v}  (K={int(_r['n_bins'])})\nimp={_r['imp_pct']:+.1f}%",
        xy=(_r["Q_mod"], _r["imp_pct"]),
        xytext=(12, 10), textcoords="offset points", fontsize=8,
        arrowprops=dict(arrowstyle="-", color=clr, lw=1.2),
    )

ax.set_xlabel("Modularity Q", fontsize=12)
ax.set_ylabel("Q-Ising improvement over LIR (%)", fontsize=12)
ax.set_title(f"Modularity vs Q-Ising improvement  (\u03c1={r_s:.2f}, p={p_s:.3f})",
             fontweight="bold")
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
savefig(fig, "qmod_vs_improvement_scatter")
plt.show()


In [ ]:
# ── Network + Q-Ising seeding sequence ────────────────────────────────────
import networkx as nx

# Villages from contrasting-pair selection (cell above)
print(f"High-Q (low imp) : village {_hi_mod_v}  ({df_feat.loc[_hi_mod_v, 'imp_pct']:+.1f}%  Q={df_feat.loc[_hi_mod_v,'Q_mod']:.3f})")
print(f"Low-Q  (high imp): village {_lo_mod_v}  ({df_feat.loc[_lo_mod_v, 'imp_pct']:+.1f}%  Q={df_feat.loc[_lo_mod_v,'Q_mod']:.3f})")

_MARKERS = ["o", "s", "^", "D", "v", "p", "h", "*", "X", "P"]
_NET_CLR  = ["#2ecc71", "#e74c3c", "#3498db", "#f39c12",
             "#9b59b6", "#1abc9c", "#e67e22", "#34495e", "#e91e63", "#00bcd4"]

def _stat_str(v):
    r = df_feat.loc[v]
    return f"N={int(r.N)}  K={int(r.n_bins)}  Q={r.Q_mod:.3f}  imp={r.imp_pct:+.1f}%  avg_deg={r.avg_degree:.1f}"

# ── Re-train Q-Ising for each showcase village ─────────────────────────
_showcase_models = {}
for _v in [_hi_mod_v, _lo_mod_v]:
    _vd = village_data[_v]
    print(f"\n{'='*44}  Village {_v}  (N={_vd['N']}, k={_vd['n_bins']})")

    _trajs_sc = []
    for _ep in range(N_TRAIN_EPS):
        _env_ep = EdgeBetCommunitySISEnv(
            _vd["adj"], _vd["beta_vec"], _vd["delta_vec"], _vd["block_labels"],
            T=T_TRAIN, seed=DATA_SEED + _ep, init_rate=INIT_RATE)
        _trajs_sc.append(_env_ep.collect_behavioral_episode(skip_prob=SKIP_PROB))

    _showcase_models[_v] = QIsingPolicy(
        ising_augmentation=True, label=f"showcase_{_v}", **_qi_kwargs
    ).fit(_trajs_sc, _vd["adj"], _vd["block_labels"])
    print(f"  CQL trained.")

# ── Record Q-Ising bin choices for one episode ────────────────────────
def _record_choices(policy_fn, vd, seed=99):
    env = EdgeBetCommunitySISEnv(
        vd["adj"], vd["beta_vec"], vd["delta_vec"], vd["block_labels"],
        T=T_TEST, seed=seed, init_rate=TEST_INIT_RATE)
    obs = env.reset(); done = False; seeds_used = 0
    choices = []
    step = 0
    while not done:
        a = policy_fn(obs, env) if seeds_used < K_SEEDS else None
        obs, _, done, _ = env.step(a)
        if a is not None and 0 <= int(a) < env.n_bins:
            choices.append((step, int(a)))
            seeds_used += 1
        step += 1
    return choices

In [ ]:
def _community_layout(G, block_labels, n_bins, seed=42):
    """Spring layout seeded with community centers on a circle.
    Guarantees communities start spatially separated regardless of modularity."""
    rng = np.random.default_rng(seed)
    angles = np.linspace(0, 2 * np.pi, n_bins, endpoint=False)
    centers = {b: (np.cos(a), np.sin(a)) for b, a in enumerate(angles)}
    init_pos = {}
    for node in G.nodes():
        b = int(block_labels[node])
        cx, cy = centers[b]
        init_pos[node] = (cx + rng.normal(0, 0.25), cy + rng.normal(0, 0.25))
    return nx.spring_layout(G, pos=init_pos, seed=seed,
                            k=1.2 / np.sqrt(len(G)), iterations=60)


In [ ]:
def _plot_village_network_seq(ax, village_id, title):
    """Network plot using community-aware layout and density-scaled edge alpha."""    
    vd  = village_data[village_id]
    G   = nx.from_numpy_array(vd["adj"])
    pos = _community_layout(G, vd["block_labels"], vd["n_bins"])
    n_edges    = G.number_of_edges()
    edge_alpha = float(np.clip(8.0 / max(n_edges / vd["N"] * 10, 1), 0.04, 0.45))
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=edge_alpha, width=0.5,
                           edge_color="#888888")
    for b in range(vd["n_bins"]):
        members = np.where(vd["block_labels"] == b)[0]
        if len(members) == 0:
            continue
        xs = np.array([pos[n][0] for n in members])
        ys = np.array([pos[n][1] for n in members])
        ax.scatter(xs, ys, marker=_MARKERS[b % len(_MARKERS)],
                   color=_NET_CLR[b % len(_NET_CLR)],
                   s=max(25, 500 / np.sqrt(vd["N"])),
                   edgecolors="white", linewidths=0.4, zorder=3,
                   label=f"Comm {b}  (n={len(members)}, β={vd['beta_vec'][members[0]]:.3f})")
    ax.set_title(title, fontsize=10, pad=6)
    ax.axis("off")
    ax.legend(loc="lower left", fontsize=7, framealpha=0.8,
              markerscale=1.2, handletextpad=0.4)


# ── 1×4 figure: [network | seeding] × [high-Q | low-Q] ───────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for col_off, (_v, row_label) in enumerate([
    (_hi_mod_v, f"HIGH Q ({df_feat.loc[_hi_mod_v,'Q_mod']:.3f})"),
    (_lo_mod_v, f"LOW Q  ({df_feat.loc[_lo_mod_v,'Q_mod']:.3f}) "),
]):
    _vd = village_data[_v]
    _k  = _vd["n_bins"]

    # Network
    _plot_village_network_seq(
        axes[col_off * 2], _v,
        f"Village {_v} — {row_label}\n{_stat_str(_v)}")

    # Q-Ising bin choices
    ax = axes[col_off * 2 + 1]
    _choices = _record_choices(_showcase_models[_v], _vd, seed=99)
    _steps_c = [c[0] for c in _choices]
    _bins_c  = [c[1] for c in _choices]

    for b in range(_k):
        chosen_at = [s for s, cb in _choices if cb == b]
        ax.scatter(chosen_at, [b] * len(chosen_at),
                   color=_NET_CLR[b % len(_NET_CLR)],
                   marker=_MARKERS[b % len(_MARKERS)],
                   s=120, zorder=3,
                   label=f"Comm {b}  (n={_vd['bin_counts'][b]})")

    ax.step(_steps_c, _bins_c, where="mid", lw=1.2,
            color="grey", alpha=0.5, zorder=2)
    ax.set_xlabel("Step", fontsize=10)
    ax.set_ylabel("Community seeded", fontsize=10)
    ax.set_title(f"Q-Ising seeding — Village {_v}", fontsize=10)
    ax.set_yticks(range(_k))
    ax.set_yticklabels([f"Comm {b}" for b in range(_k)], fontsize=8)
    ax.set_xlim(-0.5, T_TEST - 0.5)
    ax.grid(axis="x", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle(
    f"Community Structure and Q-Ising Seeding",
    fontsize=12, y=1.01)
plt.tight_layout()
savefig(fig, "network_seeding_sequence_edgebet")
plt.show()


# EMVS Fit Quality Pooled

In [ ]:
# ── EMVS ROC/AUC: separate pass over all villages ─────────────────────────
from sklearn.metrics import roc_auc_score, roc_curve

V0_E, V1_E, C_E, SIGMA_E = 0.01, 10.0, 1.0, 10.0

_common_fpr   = np.linspace(0, 1, 300)
_all_tprs     = []
_all_aucs     = []
_skipped      = 0
_emvs_all_vills = {}   # store fitted EMVS results for downstream analysis

print(f"Re-fitting EMVS for {len(VIABLE_VILLS)} villages ...")
_t_auc = _time.perf_counter()

for _v in VIABLE_VILLS:
    _vd = village_data[_v]

    # ── collect one behavioral episode (same params as main loop) ─────────
    _env_ep = EdgeBetCommunitySISEnv(
        _vd["adj"], _vd["beta_vec"], _vd["delta_vec"], _vd["block_labels"],
        T=T_TRAIN, seed=DATA_SEED, init_rate=INIT_RATE)
    _tr = _env_ep.collect_behavioral_episode(skip_prob=SKIP_PROB)

    _Xv = _tr["adopted"][:-1].astype(float)[np.newaxis]   # (1, T-1, N)
    _yv = _tr["adopted"][1:].astype(float)[np.newaxis]    # (1, T-1, N)
    _tv = _tr["seeded_nodes"][:-1][np.newaxis]            # (1, T-1)

    # ── fit EMVS ──────────────────────────────────────────────────────────
    try:
        _emvs_v = fit_group_ising_emvs(
            _Xv, _yv, _tv, _vd["adj"], _vd["block_labels"],
            v0=V0_E, v1=V1_E, c=C_E, sigma_sq=SIGMA_E, verbose=False)
        _emvs_all_vills[_v] = _emvs_v   # store for downstream analysis
    except Exception as _e:
        print(f"  Village {_v}: EMVS failed ({_e}), skipping.")
        _skipped += 1
        continue

    # ── predictions: shape (T-1, N) ───────────────────────────────────────
    _X_flat = _Xv[0]   # (T-1, N)
    _y_flat = _yv[0]   # (T-1, N)
    _T      = _X_flat.shape[0]

    _p_all = np.array([
        predict_group_ising_emvs(_X_flat[t], action=-1, result=_emvs_v, A=None)
        for t in range(_T)
    ])   # (T-1, N)

    # ── per-node ROC (skip constant nodes) ────────────────────────────────
    for j in range(_vd["N"]):
        _y_j = _y_flat[:, j]
        _p_j = _p_all[:, j]
        if _y_j.std() == 0:
            continue
        try:
            _auc_j = roc_auc_score(_y_j, _p_j)
            _fpr_j, _tpr_j, _ = roc_curve(_y_j, _p_j)
            _all_tprs.append(np.interp(_common_fpr, _fpr_j, _tpr_j))
            _all_aucs.append(_auc_j)
        except Exception:
            pass

print(f"Done ({_time.perf_counter()-_t_auc:.1f}s)  "
      f"nodes evaluated: {len(_all_aucs)}  villages skipped: {_skipped}")
print(f"Per-node AUC  mean={np.mean(_all_aucs):.4f}  std={np.std(_all_aucs):.4f}  "
      f"min={np.min(_all_aucs):.4f}  max={np.max(_all_aucs):.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────
_mean_tpr  = np.mean(_all_tprs, axis=0)
_std_tpr   = np.std(_all_tprs,  axis=0)
_macro_auc = float(np.mean(_all_aucs))

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(_common_fpr, _mean_tpr, color=POLICY_COLORS["q_ising"], lw=2.2,
        label=f"Macro-avg ROC  (AUC = {_macro_auc:.3f})")
ax.fill_between(_common_fpr,
                np.maximum(_mean_tpr - _std_tpr, 0),
                np.minimum(_mean_tpr + _std_tpr, 1),
                alpha=0.15, color=POLICY_COLORS["q_ising"],
                label=r"$\pm$1 std across nodes")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random classifier")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("EMVS Group Ising: Macro-Averaged ROC\n"
             "(pooled across all villages and nodes)", fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout()
savefig(fig, "emvs_roc_auc")
plt.show()


In [ ]:
# ── Build per-(village, bin) records from all EMVS fits ───────────────────
from scipy.stats import spearmanr

_pb_all = []
for _v, _emvs in _emvs_all_vills.items():
    _vd  = village_data[_v]
    G    = _vd["n_bins"]
    _bins = [np.where(np.asarray(_vd["block_labels"]) == b)[0]
             for b in range(G)]
    _cm  = _emvs["extracted"]["coupling_matrix_mean"]   # (G, G): _cm[g, m] = gamma_{g,m}
    for g in range(G):
        m = _bins[g]
        if len(m) == 0:
            continue
        params_g = _emvs["group_params"][g]
        # cross-group indices
        _others = [h for h in range(G) if h != g]
        _mean_in  = float(np.mean(_cm[g, _others])) if _others else float("nan")
        _mean_out = float(np.mean(_cm[_others, g])) if _others else float("nan")
        _pb_all.append({
            "village":       _v,
            "bin":           g,
            "true_beta":     float(_vd["beta_vec"][m].mean()),
            "true_delta":    float(_vd["delta_vec"][m].mean()),
            "intercept":     float(params_g[0]),
            "rho":           float(params_g[1]),
            "treat":    float(params_g[2]),
            "spillover":     float(params_g[3]),
            "self_coupling": float(params_g[4 + g]),
            "mean_ingoing":  _mean_in,
            "mean_outgoing": _mean_out,
        })

df_pb_all = pd.DataFrame(_pb_all)
_n_vills = df_pb_all["village"].nunique()
print(f"Per-bin records: {len(df_pb_all)}  across {_n_vills} villages  "
      f"(mean {len(df_pb_all)/_n_vills:.1f} bins/village)")
print(f"Single-bin villages (no cross couplings): "
      f"{(df_pb_all['mean_ingoing'].isna()).sum()} NaN rows in cross-coupling columns")

# ── Scatter helper ─────────────────────────────────────────────────────────
def _scatter_fit(ax, x, y, xlabel, ylabel, title, color):
    mask = np.isfinite(x) & np.isfinite(y)
    xm, ym = x[mask], y[mask]
    ax.scatter(xm, ym, alpha=0.45, s=22, color=color, edgecolor="k", lw=0.2, zorder=2)
    if len(xm) >= 2:
        c = np.polyfit(xm, ym, 1)
        xr = np.linspace(xm.min(), xm.max(), 200)
        ax.plot(xr, np.polyval(c, xr), color="#e74c3c", lw=1.8, zorder=3)
        r, p = spearmanr(xm, ym)
        pstr = f"{p:.2e}" if p >= 1e-300 else "< 1e-300"
        stat_str = f"Spearman r = {r:.3f}   p = {pstr}"
    else:
        stat_str = "n < 2"
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(f"{title}\n{stat_str}", fontsize=9, fontweight="bold")
    ax.grid(alpha=0.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# ── Figure 1: original 3 panels (rho, gamma_gg, beta_treat) ───────────────
fig1, axes1 = plt.subplots(1, 3, figsize=(15, 4.5))
_scatter_fit(
    axes1[0],
    df_pb_all["true_delta"].values, df_pb_all["rho"].values,
    r"True Churn Rate (per bin)",
    r"EMVS Persistence",
    r"Recovery captured: persistence vs churn rate",
    color="#27ae60")
_scatter_fit(
    axes1[1],
    df_pb_all["true_beta"].values, df_pb_all["self_coupling"].values,
    r"True Spread rate (per bin)",
    r"EMVS $\gamma_{g,g}$ (within-group coupling)",
    r"Spread captured: $\gamma_{g,g}$ vs Spread Rate",
    color="#2980b9")
_scatter_fit(
    axes1[2],
    df_pb_all["true_beta"].values, df_pb_all["treat"].values,
    r"True Spread rate (per bin)",
    r"EMVS Direct Treatment (seeding effect)",
    r"Seeding effect vs Spread rate:",
    color="#8e44ad")
fig1.suptitle(
    f"EMVS Parameters vs. True SIS Parameters  "
    f"(all {_n_vills} villages, {len(df_pb_all)} bin obs.)",
    fontsize=11, fontweight="bold", y=1.02)
plt.tight_layout()
savefig(fig1, "emvs_vs_true_params_all_villages")
plt.show()

# ── Figure 2: intercept, spillover, mean ingoing/outgoing coupling ─────────
fig2, axes2 = plt.subplots(1, 5, figsize=(22, 4.5))
_scatter_fit(
    axes2[0],
    df_pb_all["true_beta"].values, df_pb_all["intercept"].values,
    r"True Spread rate",
    r"EMVS intercept",
    r"Intercept vs Spread Rate",
    color="#c0392b")
_scatter_fit(
    axes2[1],
    df_pb_all["true_delta"].values, df_pb_all["intercept"].values,
    r"True Recovery Rate",
    r"EMVS intercept",
    r"Intercept vs Recovery Rate",
    color="#e67e22")
_scatter_fit(
    axes2[2],
    df_pb_all["true_beta"].values, df_pb_all["spillover"].values,
    r"True Spread rate",
    r"EMVS spillover",
    r"Spillover vs Spread Rate",
    color="#16a085")
_scatter_fit(
    axes2[3],
    df_pb_all["true_beta"].values, df_pb_all["mean_ingoing"].values,
    r"True Spread rate",
    r"EMVS mean ingoing coupling\n(cross-group, excl. self)",
    r"Mean ingoing $\bar\gamma_{\cdot\to g}$ vs Spread Rate",
    color="#2471a3")
_scatter_fit(
    axes2[4],
    df_pb_all["true_beta"].values, df_pb_all["mean_outgoing"].values,
    r"True Spread rate",
    r"EMVS mean outgoing coupling\n(cross-group, excl. self)",
    r"Mean outgoing $\bar\gamma_{g\to\cdot}$ vs Spread Rate",
    color="#884ea0")
fig2.suptitle(
    f"Additional EMVS Parameters vs. True SIS Parameters  "
    f"(all {_n_vills} villages)",
    fontsize=11, fontweight="bold", y=1.02)
plt.tight_layout()
savefig(fig2, "emvs_vs_true_params_extra_all_villages")
plt.show()
